---
# **MACHINE LEARNING**
---

```text
1. Data Preprocessing
2. Train-Test SPLIT
3. Feature Engineering
4. Imbalance handling
5. Compare models
6. Select a model
7. Train Model
8. Evaluate Model
9. Improve Model
10. Deploy Model
```

<hr>

#### 🧰 INSTALLs


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: red;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

In [ ]:
# install numpy and pandas if not already installed
#!pip install numpy pandas

# install xgboost if not already installed
#!pip install xgboost

# install shap if not already installed
#!pip install shap

# install imbalanced-lear
!pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]



<hr>

#### 📂 IMPORTs


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: red;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

from xgboost import XGBClassifier

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)

---
## **Data Preprocessing**
---

### **Load & View the data**

In this project, we will be working with Online Shoppers Intention UCI Machine Learning. The data can be found here:

https://www.kaggle.com/datasets/henrysue/online-shoppers-intention/data

Metadata



In [ ]:
import pandas as pd

# load the clean dataset
df = pd.read_csv("../data/processed/online_shoppers_intention_01_standard.csv")
print("Dataset loaded successfully. Shape:", df.shape)
display(df.head())

In [ ]:
display(df.info())

### **Check shape**

In [ ]:
print("Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")

### **Check data types**

In [ ]:
print("Data Types:")
df.dtypes

### **Check missing values**

There are multiple strategies to handle missing data
- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

In [ ]:
print("Missing Values:")
print(df.isna().sum())
print(df.info())

In [ ]:
# Loop print unique values of each column
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")


In [ ]:
# loop through columns and print sum of nulls for each column
for col in df.columns:
    print(f"Null values in '{col}': [{df[col].isna().sum()}]")

In [ ]:
# compare the number of rows with the number of nulls in each column 
# to identify columns with a high percentage of missing values
print("Null values and percentage of nulls for each column:")
for col in df.columns:
    num_rows = df.shape[0]
    num_nulls = df[col].isna().sum()
    percent_nulls = (num_nulls / num_rows) * 100
    print(f"{col}: {num_nulls} nulls ({percent_nulls:.2f}%)")

### **Check duplicates**

In [ ]:
# print number of duplicated rows
num_duplicates = df.duplicated().sum()
print(f"Number of duplicated rows: {num_duplicates}")

### **Leakage Handling**
**DROP**
- identifier column (Id) and (Name), they are not useful features (high cardinality and not predictive)
- columns that are future information

Data leakage happens when:
- Information from the test set influences training
- Or when a feature contains **future information**
- Or when preprocessing is done **before splitting**

Very important:
- Remove post-event features
- Remove target proxies
- Remove ID-like features
- Check correlation with target
- Drop leakage features  
- Ensure no post-target info (variables) exists
- Fit transformations only on training set

Note:
- This step is more important than imbalance handling
- This step can be done either before or right after the step 2 - train/test split

In [ ]:
# df = df.drop(["IDs", "Name",...], axis=1)

### **Deal with Nulls & Duplicates**

In [ ]:
# remove nulls
# df = df.dropna()

# remove duplicates
# df = df.drop_duplicates()


### **Convert Data TYPE**

#### Convert Boolean to int

In [ ]:
# boolean columns
bool_cols = ["weekend", "revenue"]

# convert booleans to integers (True=1, False=0) for better modeling
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   admin                  12330 non-null  int64  
 1   admin_duration         12330 non-null  float64
 2   info                   12330 non-null  int64  
 3   info_duration          12330 non-null  float64
 4   prod_related           12330 non-null  int64  
 5   prod_related_duration  12330 non-null  float64
 6   bounce_rate            12330 non-null  float64
 7   exit_rate              12330 non-null  float64
 8   page_value             12330 non-null  float64
 9   special_day            12330 non-null  float64
 10  month                  12330 non-null  str    
 11  os                     12330 non-null  int64  
 12  browser                12330 non-null  int64  
 13  region                 12330 non-null  int64  
 14  traffic_type           12330 non-null  int64  
 15  visitor_type 

---
### **CHECK PREPROCESSED DATA**

| Variable      | Cat vs Num | Identity   | Original Python Type | Action   | Convert to | Definition |
|---------------|------------|------------|------------|----------|------------|------------|
|   | ID         | Identifier | object     | DROP     | —          | |


In [ ]:
# display cleaned dataframe
display(df.head())

In [ ]:
# display info cleaned dataframe
df.info()

In [ ]:
# loop print unique values for each column
for col in df.columns:
        print(f"['{col}]': {df[col].unique()}")

---
## **Train-Test Split**

Now that we have split the data into **features** and **target** variables and imported the **train_test_split** function, split X and y into X_train, X_test, y_train, and y_test. 80% of the data should be in the training set and 20% in the test set.

### **Define target (y)**

In [ ]:
# define target variable
y = df["revenue"]

### **Define features (X)**
```text
X = df.drop("revenue", axis=1)  # features without the target variable "revenue"
y = df["revenue"]               # target variable "revenue"
```

In [ ]:
# define features
X = df.drop(["revenue"], axis=1)

### **SPLIT**
Use `random_state=42`, `stratify`

```text
X,                  # features without the target variable "revenue"
y,                  # target variable "revenue"
test_size=0.2,      # 20% test set, 80% train set
stratify=y,         # maintain class balance in splits
random_state=42     # for reproducibility, we can choose any integer as the random state
```

In [ ]:

#split the data into training and testing sets (stratify by target variable to maintain class distribution)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#display the shapes of the resulting sets
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

**Check split data**

In [ ]:
print(f"Features=(X) and target=(y) separated. \nShapes:\nX =", X.shape, ", y =", y.shape)
print(100*"-")
print(f"Train/Test split completed. \nShapes: \nX_train = {X_train.shape}, y_train = {y_train.shape}, \nX_test = {X_test.shape}, y_test = {y_test.shape}")
print(100*"-")
print("Display X_train:")
display(X_train.head())

---
## **Feautres Engineering**

Steps to Features Engineering:

1. Identify Categorical vs Numerical Features

2. Encode Categorical Features & Scale Numerical Features
- Categorical Features:
    - Hot-one encoding for seperate categories
    - Labeling for ordinal variables
- Numerical Features:
    - Scaling because models can be sensitive to big numbers

3. Convert Target to numeric python type int (False=0; True=1) //Can also be done in the data prep
4. Convert all Features to numeric python type int or float //Can also be done in the data prep
5. Save `X_train.csv`,`X_test.csv` and `y_train.csv`,`y_test.csv`

### **1. Identify Categorical vs Numerical Features**

In [ ]:
# define categorical features
cat_cols = [""]

# define numerical features
num_cols = [""]

### **2. Encoding Cat & Scaling Num Features**

**All Models require Encoding Categorical Features** 
- OneHotEncoder (nominal)
- OrdinalEncoder (ordinal)
- Handle unknown categories when Encoding
- Avoid label encoding for nominal variables

**Scaling can be conditional per model:**
- **Models that require Scaling:** 
    - Linear models (Logistic Regression, Ridge, Lasso, ElasticNet)
    - Support Vector Machines (SVM)
    - K-Nearest Neighbors (KNN)
    - Neural Networks
    - K-Means / distance-based algorithms

- **Models that DON'T require Scaling:** 
    - Tree-based models (Decision Trees, Random Forest, XGBoost, LightGBM, CatBoost)



In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ---- One-Hot Encode ----
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# categorical features in X_train
X_train_cat = pd.DataFrame(
    ohe.fit_transform(X_train[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_train.index
)
# categorical features in X_test
X_test_cat = pd.DataFrame(
    ohe.transform(X_test[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=X_test.index
)

# ---- Scale Numerical ----
scaler = StandardScaler() 
# StandardScaler scales numerical features to have mean=0 and std=1

# numerical features in X_train
X_train_num = pd.DataFrame(
    scaler.fit_transform(X_train[num_cols]),
    columns=num_cols,
    index=X_train.index
)

# numerical features in X_test
X_test_num = pd.DataFrame(
    scaler.transform(X_test[num_cols]),
    columns=num_cols,
    index=X_test.index
)

# ---- Replace original X_train and X_test ----
X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

# display the shapes of the resulting sets
print("X_train shape after encoding and scaling:", X_train.shape)
display(X_train.head())
print("X_test shape after encoding and scaling:", X_test.shape)
display(X_test.head())

print("y_train shape:", y_train.shape)
display(y_train.head())
print("y_test shape:", y_test.shape)
display(y_test.head())

# ---- Save preprocessed data to CSV files ----
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)


---
## **Imbalance handling**

**Target distribution**, Class balance percentages:
- Class True  = 1 :  /  ≈ 15-16%
- Class False = 0 :  /  ≈ 84-85%
 check (imbalanced 84%/16%)

In [ ]:
# check class imbalance in target variable
print("Class distribution in y_train:")
print(y_train.value_counts())

**Important decision!** Which to Use:
- SMOTE?
- Random under-sampling?
- class_weight?
- threshold tuning?

⚠️ VERY IMPORTANT:
Apply SMOTE only on training set inside cross-validation fold. Otherwise → leakage.

In [ ]:
class_weight='balanced'

#### Option A — Use Class Weights (Most Standard & Clean) 

#### Option B — Resampling (Use Carefully)

#### Option C — Threshold Tuning (Often Overlooked) 

---
# **Select Model (XGBoost)**

XGBoost has a built-in way to handle class imbalance using the scale_pos_weight parameter.

---

### **Load Train/Test CSVs**

In [ ]:
import pandas as pd
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")   
y_train = pd.read_csv("../data/processed/y_train.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")

### **SANITY CHECK**

In [ ]:
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print(100*"-")

print("Class balance:")
print(pd.Series(y_train).value_counts(normalize=True))
print(100*"-")

### **Define Model**

In [ ]:
from sklearn.neighbors import XGBClassifier

# define XGBoost model
model = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=3, 
    random_state=42
)

### **Predict**

In [ ]:
# PREDICT
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

### **Train Model**

**Purpose** →  Train the selected algorithm on the training data to learn patterns and relationships between features and the target variable. 

In [ ]:
# fit the model to the training data
model.fit(X_train, y_train)

### **Evaluate Model**

Evaluate the XGBOOST model on the test data

**Metrics:** 
- `Accuracy`, // this metric is not reliable
- `Classification Report`, 
- `PR/ROC - AUC`, 
- `Confusion Matrix`

**🎯 Metrics to Prioritize:**

1️⃣ Test ROC AUC (Most Important)
- Measures ranking quality
- Threshold independent
- Best overall indicator of model quality

2️⃣ Test Accuracy
- Easy interpretability
- Good when classes are balanced (yours are balanced)

3️⃣ Overfitting Gap (Train vs Test)
- Small gap = better generalization
- Large gap = overfitting
- PR AUC and F1 were supportive but secondary.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)
import xgboost

# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("XGBoost Accuracy:", accuracy)

# Classification Report
print("XGBoost Classification Report:\n")
print(classification_report(y_test, y_pred))

# Precision-Recall AUC
pr_auc = average_precision_score(y_test, y_proba)
print("XGBoost PR AUC:", pr_auc)

# ROC AUC
roc_auc = roc_auc_score(y_test, y_proba)
print("XGBoost ROC AUC:", roc_auc)

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("XGBoost Confusion Matrix:\n", conf_matrix)

### **Precision-Recall Curve PLOT**

In [ ]:

plt.figure()
plt.plot(recall, precision, label=f"PR AUC = {pr_auc:.3f}")
plt.hlines(baseline, 0, 1, linestyles="dashed", label="Baseline")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (XGBoost - Test Set)")
plt.legend()
plt.grid()
plt.show()

---
### **Check whether Model is over/under - fitting**

- **FITTING CORRECTLY:** Small difference → model generalizes well.
- **OVERFITTING:** Large gap → model memorized training data. Rule of Thumb → If: Train − Test > 5–7% probably overfitting.
- **UNDERFITTING:** Both low → model too simple.

### Method 1 to check: Compare ROC AUC Train vs Test (More reliable than accuracy)

In [ ]:
from sklearn.metrics import roc_auc_score

train_roc = roc_auc_score(y_train, xgboost.predict_proba(X_train)[:,1])
test_roc  = roc_auc_score(y_test, xgboost.predict_proba(X_test)[:,1])

print("Train ROC AUC:", train_roc)
print("Test ROC AUC :", test_roc)

### Method 2 to check: Compare Train Accuracy vs Test Accuracy (Less reliable than ROC AUC)

In [ ]:
# Train accuracy
train_acc = xgboost.score(X_train, y_train)

# Test accuracy
test_acc = xgboost.score(X_test, y_test)

print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)

---
### **CROSS VALIDATION**

- If CV score ≈ test score → good.
- If CV score much lower than train → overfitting.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(xgboost, X_train, y_train, cv=5)
print("Cross-validation mean accuracy:", scores.mean())

---
## **IMPROVE MODEL**
**Purpose** → Optimize model performance and improve generalization by refining model settings and feature selection.

#### **Adjust Decision Threshold**

- **Instead of 0.5 → ?**
- **This should improve recall.**

In [ ]:
# plot for adjusting decision threshold before and after 

#### **Feature Importance Analysis**
**For RF / XGBoost: Use  `.feature_importances_`**

#### **Feature Selection**

#### **Reducing overfitting or underfitting**

---
## **DEPLOY MODEL**
Using `Streamlit` for an interface for Real Time Prediction

---
## **NOTES**

👉 Best model by ROC-AUC: XGBoost (0.9275)
- It has the highest ability to rank positive cases above negative cases.

⚠️ Is ROC-AUC enough for imbalanced data? Short answer: No, it is not enough.
- ROC-AUC can be misleading for imbalanced datasets because:
    - It evaluates performance across all classification thresholds.
    - It gives equal importance to True Positive Rate (TPR) and False Positive Rate (FPR).
    - When negatives heavily outnumber positives, the FPR can look artificially small, inflating ROC-AUC.
    - In highly imbalanced data, a model can achieve high ROC-AUC but still perform poorly at identifying the minority class.

🔎 For imbalanced datasets, prioritize:

1️⃣ Precision-Recall AUC (PR-AUC) ⭐ (Very Important)
- More informative when the positive class is rare.
- Focuses only on the positive class performance.

2️⃣ F1-score
- Harmonic mean of precision and recall.
- Good when you need balance between false positives and false negatives.

3️⃣ Recall (Sensitivity)
- Important if missing positives is costly (e.g., fraud, disease).

4️⃣ Confusion Matrix
- Shows actual classification behavior at your chosen threshold.


📊 Recommended Evaluation Metrics (Very Important)

Because the dataset is IMBALANCED:

✅ ROC-AUC
✅ Precision
✅ Recall
✅ F1-score
❌ Avoid relying only on accuracy